# **1. Import Library & Environment Setup**

In [30]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import classification_report
import joblib

# **2. Data Loading**


In [31]:
df = pd.read_csv("data_clustering.csv")

In [32]:
df.head()

,TransactionAmount,TransactionType,Location,Channel,CustomerAge,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance,Target
0,-1.111922,1,36,0,1.426636,0,-0.541568,0.0,0.002918,1
1,0.546926,1,15,0,1.313889,0,0.308502,0.0,2.216531,0
2,-0.597984,1,23,2,-1.448403,3,-0.895763,0.0,-1.018513,1
3,-0.331350,1,33,2,-1.053790,3,-1.334965,0.0,0.887895,1
4,-0.754364,1,28,0,-1.504776,3,0.747704,0.0,-1.105726,1


# **3. Feature Encoding (One-hot Encoding)**

In [33]:
categorical_cols = list(df.select_dtypes(include=['object']).columns)

df_encoded = pd.get_dummies(
    df,
    columns = categorical_cols,
    drop_first = True
)

df_encoded.head()

,TransactionAmount,TransactionType,Location,Channel,CustomerAge,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance,Target
0,-1.111922,1,36,0,1.426636,0,-0.541568,0.0,0.002918,1
1,0.546926,1,15,0,1.313889,0,0.308502,0.0,2.216531,0
2,-0.597984,1,23,2,-1.448403,3,-0.895763,0.0,-1.018513,1
3,-0.331350,1,33,2,-1.053790,3,-1.334965,0.0,0.887895,1
4,-0.754364,1,28,0,-1.504776,3,0.747704,0.0,-1.105726,1


# **4. Data Splitting**

In [34]:
X = df_encoded.drop('Target', axis=1)
y = df_encoded['Target']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42,
    stratify = y
)

print("Jumlah data total: ",len(X))
print("Jumlah data latih: ",len(X_train))
print("Jumlah data test: ",len(X_test))

Jumlah data total:  1945
Jumlah data latih:  1556
Jumlah data test:  389


# **5. Base Modeling (Decision Tree)**

In [35]:
decision_tree_model = DecisionTreeClassifier(random_state=42)
decision_tree_model.fit(X_train, y_train)

DecisionTreeClassifier(random_state=42)

In [36]:
joblib.dump(decision_tree_model, 'decision_tree_model.h5')

['decision_tree_model.h5']

In [37]:
new_model = RandomForestClassifier(random_state=42)
new_model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [38]:
# Menampilkan hasil evaluasi akurasi, presisi, recall, dan F1-Score.

y_pred_dt = decision_tree_model.predict(X_test)
y_pred_new = new_model.predict(X_test)

print("Decision Tree Performance")
print(classification_report(y_test, y_pred_dt))

print("="*50)

print("New Model Performance")
print(classification_report(y_test, y_pred_new))

Decision Tree Performance
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       196
           1       1.00      1.00      1.00       193

    accuracy                           1.00       389
   macro avg       1.00      1.00      1.00       389
weighted avg       1.00      1.00      1.00       389

New Model Performance
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       196
           1       1.00      1.00      1.00       193

    accuracy                           1.00       389
   macro avg       1.00      1.00      1.00       389
weighted avg       1.00      1.00      1.00       389



In [39]:
joblib.dump(new_model, 'explore_random_forest_model_classification.h5')

['explore_random_forest_model_classification.h5']

# **6. Hyperparameter Tuning (Random Forest)**

In [40]:
# Hyperparameter Tuning dan Latih ulang.

params = {'n_estimators': [50, 100, 200],
          'max_depth': [None, 10, 20],
          'min_samples_split': [2, 5, 10]}

new_model_tuned = GridSearchCV(
    estimator = RandomForestClassifier(random_state=42),
    param_grid = params,
    cv = 5,
    scoring = 'accuracy'
)

new_model_tuned.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42),
             param_grid={'max_depth': [None, 10, 20],
                         'min_samples_split': [2, 5, 10],
                         'n_estimators': [50, 100, 200]},
             scoring='accuracy')

In [41]:
# Menampilkan hasil evaluasi akurasi, presisi, recall, dan F1-Score pada algoritma yang sudah dituning.

y_pred_tuning = new_model_tuned.predict(X_test)

print("Tuned Model Performance")
print(classification_report(y_test, y_pred_tuning))

Tuned Model Performance
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       196
           1       1.00      1.00      1.00       193

    accuracy                           1.00       389
   macro avg       1.00      1.00      1.00       389
weighted avg       1.00      1.00      1.00       389



In [42]:
# Menyimpan Model hasil tuning

joblib.dump(new_model_tuned, 'tuning_classification.h5')

['tuning_classification.h5']

# **7. Kesimpulan**
Kedua model klasifikasi (Decision Tree dan Random Forest) berhasil mempelajari pola segmentasi pelanggan secara sempurna (Akurasi, Presisi, Recall, dan F1-Score: 1.00). Tingginya metrik evaluasi ini merupakan hasil yang wajar dan logis, karena fitur target (`Target`) secara matematis diturunkan dari fitur prediktor melalui algoritma K-Means Clustering pada tahap sebelumnya.

Model `RandomForestClassifier` yang telah dioptimasi (*tuning*) kini siap di-*deploy* (`tuning_classification.h5`) untuk mengklasifikasikan segmen nasabah baru secara otomatis tanpa perlu memproses ulang algoritma *clustering*.

End of Code